# 09. 텍스트 임베딩 3종 비교 — 누가 한국어 규정을 잘 찾나

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/acho98/gunyang-ai-api/blob/main/labs/09_text_embeddings_compare.ipynb)

> 🧭 **스토리** — 규정 도우미의 첫 부품은 **검색기**다. 질문을 받으면 규정집 조각(청크) 13개 중 근거가 되는 조각을 골라야 한다. 후보 임베딩 모델은 셋: 작고 빠른 `multilingual-e5-small`, 한국어로 다듬은 `KURE-v1`, 이미지까지 다루는 신상 `EmbeddingGemma 2`. 벤치마크 표 말고 **우리 문서와 우리 질문**으로 직접 겨뤄 본다.
>
> 📖 **이론 연결** — 이론2 p61~70(임베딩·유사도·모델 비교·벤치마크·접두어) · ⏱ **15분** · 🖥 **GPU 권장**
>
> GPU(T4)가 있으면 빠릅니다. 없어도 CPU로 돌아갑니다(느림).
> 💡 **앞 노트북을 열어 둔 상태라면** 그 노트북에서 `런타임 → 런타임 연결 해제 및 삭제`를 먼저 하세요. 무료 코랩은 GPU 세션을 하나만 허용합니다.

**셀 배지**: ⚙️ 준비 · 🔵 따라 실행 · 🟡 바꿔서 실행 · 🤔 생각해보기 · 🟠 심화(시간 남을 때)

## 0. 준비 ⚙️
`sentence-transformers` 6.1 이상이 필요합니다(EmbeddingGemma 2 지원).
모델 세 개를 합쳐 약 4GB를 내려받습니다(코랩 기준 1~2분). 설치 중 빨간 `ERROR: pip's dependency resolver …` 문구가 보여도, 코랩에 미리 깔린 다른 패키지(gradio · google-colab 등)와 버전이 달라 나오는 안내라 실습에는 영향이 없습니다.

In [ ]:
%pip -q install -U sentence-transformers transformers pillow

In [ ]:
# ⚙️ GPU 확인 — T4가 잡혔는지, 메모리가 얼마인지
import torch, subprocess
HAS_GPU = torch.cuda.is_available()
if HAS_GPU:
    name = torch.cuda.get_device_name(0)
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU: {name} · 메모리 {total_gb:.1f} GB")
else:
    print("GPU 없음 — CPU로 진행합니다(느립니다). 런타임 유형에서 T4를 선택하면 빨라집니다.")
DEVICE = "cuda" if HAS_GPU else "cpu"
DTYPE = torch.float16 if HAS_GPU else torch.float32   # T4는 bf16을 지원하지 않아 fp16을 씁니다

In [ ]:
# ⚙️ 실습 데이터 로더 — 깃허브(gunyang-ai-api/data)에서 자동으로 받아옵니다. 업로드 불필요.
import os, json, urllib.request
from pathlib import Path
DATA_BASE = os.environ.get("GENAI_BASE", "https://raw.githubusercontent.com/acho98/gunyang-ai-api/main/data/")
CACHE = Path("genai_data"); CACHE.mkdir(exist_ok=True)

def fetch(name: str) -> str:
    """파일 이름(예: 'hanbit_rules.pdf', 'scan/p2.png') → 로컬 경로. URL이면 내려받아 캐시."""
    if DATA_BASE.startswith("http"):
        dst = CACHE / name
        if not dst.exists():
            dst.parent.mkdir(parents=True, exist_ok=True)
            urllib.request.urlretrieve(DATA_BASE + name, dst)
        return str(dst)
    return str(Path(DATA_BASE) / name)

def load_json(name: str):
    with open(fetch(name), encoding="utf-8") as f:
        return json.load(f)

def load_text(name: str) -> str:
    with open(fetch(name), encoding="utf-8") as f:
        return f.read()

# 한글 폰트(차트용) — gunyang-data의 BMDOHYEON.ttf 를 자동으로 받아 적용. 실패해도 진행.
def set_korean_font():
    import matplotlib.pyplot as plt
    from matplotlib import font_manager as fm
    font_url = "https://raw.githubusercontent.com/acho98/gunyang-data/main/BMDOHYEON.ttf"
    dst = CACHE / "BMDOHYEON.ttf"
    try:
        if not dst.exists():
            local = Path(DATA_BASE).parent / "BMDOHYEON.ttf" if not DATA_BASE.startswith("http") else None
            if local and local.exists():
                dst.write_bytes(local.read_bytes())
            else:
                urllib.request.urlretrieve(font_url, dst)
        fm.fontManager.addfont(str(dst))
        plt.rcParams["font.family"] = fm.FontProperties(fname=str(dst)).get_name()
        plt.rcParams["axes.unicode_minus"] = False
    except Exception as e:
        print("(안내) 한글 폰트 설정 실패 — 영문 라벨은 정상:", e)
set_korean_font()
print("데이터 소스:", DATA_BASE)

In [ ]:
# ⚙️ 청크 13개와 평가 질문 15개를 불러오고, 질문마다 '정답 청크 id'를 붙입니다
import pandas as pd
chunks = load_json("hanbit_chunks_gt.json")   # 장(본문)·표 단위로 미리 잘라 둔 정답 청크
qa = load_json("hanbit_qa.json")
chunk_ids = [c["id"] for c in chunks]
chunk_texts = [c["text"] for c in chunks]

def gold_ids(item):
    """근거 청크 id 집합. 표 질문은 s{장}_{표}, 본문 질문은 s{장}_text. '표+본문'은 둘 다 정답으로 인정."""
    if item["type"] == "unanswerable":
        return set()
    text_id = f"s{item['section']}_text"
    if item["table"] is None:
        return {text_id}
    table_id = f"s{item['section']}_{item['table']}"
    return {table_id, text_id} if item["type"] == "table+text" else {table_id}

evalset = [q for q in qa if q["type"] != "unanswerable"]   # 근거가 있는 13문항으로 검색 평가
for q in evalset:
    q["gold"] = gold_ids(q)
print(f"청크 {len(chunks)}개 · 평가 질문 {len(evalset)}개 (근거 없음 {len(qa)-len(evalset)}개는 11번 노트북에서)")
display(pd.DataFrame([{"id": c["id"], "kind": c["kind"], "글자수": len(c["text"]), "앞부분": c["text"][:40].replace("\n", " ")} for c in chunks]))

## 1. 모델 세 개 불러오기 🔵
| 모델 | 크기 | 차원 | 특징 | 사용법 |
|---|---|---|---|---|
| `intfloat/multilingual-e5-small` | 118M | 384 | 100개 언어, 작고 빠름 | 질문 앞 `"query: "`, 문서 앞 `"passage: "` **필수** |
| `nlpai-lab/KURE-v1` | 568M | 1024 | bge-m3를 한국어 검색으로 파인튜닝(고려대) | 접두어 없음 |
| `google/embeddinggemma-2` | 744M | 768 (MRL: 앞 512·256·128차원만 잘라 써도 되게 학습) | 텍스트·이미지·영상·음성 한 공간, 8K 문맥 | `prompt_name="SearchQuery"` / `"Document"` |

EmbeddingGemma 2는 **float16을 쓰면 안 됩니다**(카드 경고, 값이 깨짐). T4는 bf16이 없으므로 **float32**로 올립니다.
오늘은 텍스트만 쓰므로 `config_kwargs`로 비전·음성 부품을 빼고 가볍게 띄웁니다(10번에서 전부 켭니다).

In [ ]:
import time, torch
from sentence_transformers import SentenceTransformer

SPECS = {
    "e5-small":  dict(name="intfloat/multilingual-e5-small", kw={}),
    "KURE-v1":   dict(name="nlpai-lab/KURE-v1", kw={}),
    "EG2":       dict(name="google/embeddinggemma-2",
                      kw=dict(config_kwargs={"vision_config": None, "audio_config": None})),   # 텍스트 전용으로 가볍게
}
models, info = {}, {}
for key, s in SPECS.items():
    t0 = time.time()
    m = SentenceTransformer(s["name"], device=DEVICE, model_kwargs={"dtype": torch.float32}, **s["kw"])
    n_params = sum(p.numel() for p in m.parameters())
    models[key] = m
    info[key] = {"로드(초)": round(time.time() - t0, 1), "파라미터(M)": round(n_params / 1e6),
                 "차원": m.get_embedding_dimension(), "fp32 메모리(GB)": round(n_params * 4 / 1024**3, 2)}
    print(f"{key:<9} 로드 {info[key]['로드(초)']:>5}초 · 파라미터 {info[key]['파라미터(M)']:>4}M · 차원 {info[key]['차원']}")

💡 **해석** — 파라미터 수가 곧 메모리입니다. float32는 파라미터 하나에 4바이트이므로 강사 환경에서 e5-small 118M(0.44GB), KURE-v1 568M(2.1GB), EG2 271M(1.0GB)이 찍혔습니다. 세 개를 동시에 올려도 T4(15GB)에는 넉넉합니다.
EG2는 비전(170M)·음성(300M) 부품을 빼고 띄웠기 때문에 카드의 740M이 아니라 텍스트 부분 271M만 올라갔습니다. 첫 실행의 로드 시간에는 다운로드 시간이 포함됩니다.

## 2. 문장 8개 유사도 히트맵 🔵
**같은 뜻, 다른 단어**(출장비·여비·교통비)를 모델이 얼마나 가깝게 보는지 확인합니다. 마지막 두 문장은 S1·S4의 영어 번역입니다.
모델마다 사용법이 다르므로 `embed_query` / `embed_docs` 두 함수로 감쌉니다. 이 함수가 **접두어·프롬프트를 빠뜨리지 않게 하는 안전장치**입니다.

In [ ]:
def embed_query(key, texts, **kw):
    m = models[key]
    if key == "e5-small":
        return m.encode(["query: " + t for t in texts], normalize_embeddings=True, **kw)
    if key == "EG2":
        return m.encode(texts, prompt_name="SearchQuery", normalize_embeddings=True, **kw)
    return m.encode(texts, normalize_embeddings=True, **kw)

def embed_docs(key, texts, **kw):
    m = models[key]
    if key == "e5-small":
        return m.encode(["passage: " + t for t in texts], normalize_embeddings=True, **kw)
    if key == "EG2":
        return m.encode(texts, prompt_name="Document", normalize_embeddings=True, **kw)
    return m.encode(texts, normalize_embeddings=True, **kw)

sents = ["출장비는 복귀 후 7일 이내에 정산해야 한다.",          # S1
         "여비는 출장에서 돌아온 뒤 일주일 안에 청구한다.",       # S2 = S1의 의역
         "교통비 영수증은 정산서에 첨부해야 한다.",               # S3 출장 관련, 다른 내용
         "연차휴가는 근속연수에 따라 부여한다.",                 # S4
         "휴가 일수는 회사에 다닌 햇수가 늘수록 많아진다.",       # S5 = S4의 의역
         "직무 교육비는 연 100만 원까지 지원한다.",              # S6
         "Travel expenses must be settled within 7 days after returning.",   # S7 = S1 영어
         "Annual leave is granted according to years of service."]           # S8 = S4 영어
labels = [f"S{i+1}" for i in range(len(sents))]

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
sims = {}
for ax, key in zip(axes, models):
    E = embed_docs(key, sents)                      # 같은 종류끼리 비교하므로 문서용으로 통일
    S = models[key].similarity(E, E).numpy()        # 코사인 유사도 행렬 (8×8)
    sims[key] = S
    ax.imshow(S, vmin=0, vmax=1, cmap="Blues")
    for i in range(len(S)):
        for j in range(len(S)):
            ax.text(j, i, f"{S[i, j]:.2f}", ha="center", va="center", fontsize=7,
                    color="white" if S[i, j] > 0.7 else "black")
    ax.set_xticks(range(len(labels)), labels); ax.set_yticks(range(len(labels)), labels)
    ax.set_title(key)
plt.suptitle("문장 8개 코사인 유사도 (진할수록 가깝다)"); plt.tight_layout(); plt.show()

pairs = {"S1–S2 의역": (0, 1), "S4–S5 의역": (3, 4), "S1–S7 한↔영": (0, 6), "S4–S8 한↔영": (3, 7), "S1–S6 무관": (0, 5)}
display(pd.DataFrame({k: {p: round(float(sims[k][i, j]), 3) for p, (i, j) in pairs.items()} for k in sims}))

💡 **해석** — 강사 환경 측정값:

| 문장 쌍 | e5-small | KURE-v1 | EG2 |
|---|---|---|---|
| S1–S2 의역(출장비↔여비) | 0.89 | 0.65 | 0.84 |
| S4–S5 의역(연차↔쉬는 날) | 0.90 | 0.64 | 0.81 |
| S1–S7 한↔영 | 0.82 | 0.88 | 0.83 |
| S1–S6 무관(출장비↔교육비) | **0.87** | 0.44 | 0.70 |

- 점수의 절대값은 모델마다 기준이 다릅니다. e5는 무관한 S1–S6도 0.87로, 의역 쌍(0.89)과 거의 같습니다. 점수가 0.8~0.9 사이에 몰려 있어 **구분 폭이 좁습니다**. 그래서 "0.8 넘으면 관련 있음" 같은 고정 임계값은 모델을 바꾸면 쓸 수 없습니다.
- 좋은 검색 모델은 "의역 쌍 − 무관한 쌍" **간격**이 큽니다. S1–S2 빼기 S1–S6을 계산하면 e5 0.02, KURE 0.21, EG2 0.14입니다. KURE가 '여비=출장비'와 '교육비≠출장비'를 가장 뚜렷하게 가릅니다.
- 한↔영 번역 쌍이 무관한 쌍보다 높게 나오면 다국어 공간이 잘 정렬된 것입니다. 한국어 질문으로 영어 문서를 찾을 수 있다는 뜻입니다. 단, e5는 S1–S7 한↔영(0.82)이 무관한 S1–S6(0.87)보다 낮습니다. 여기서도 절대값보다 간격을 봐야 합니다.

## 3. 검색 평가 — hit@k 🔵
청크 13개를 **문서**로, 질문을 **질의**로 임베딩한 뒤 코사인 유사도로 줄을 세웁니다.
**hit@k** = 정답 청크가 상위 k개 안에 들어간 질문의 비율. RAG에서는 LLM에게 k개를 넘기므로 "k개 안에만 있으면 된다"는 실무 지표입니다(이론2 p95).
**MRR** = 정답 순위의 역수 평균(1등이면 1, 2등이면 0.5). hit@k보다 순위 차이에 민감합니다. 계산 예는 이론2 p66(정답 1개)·p67(정답이 여러 개일 때 Recall@k·nDCG@k)에 있습니다.

질문 세트는 두 가지입니다.
- **규정 질문 13개**: `hanbit_qa.json`의 근거 있는 문항. 규정집 용어를 그대로 씁니다("국내 출장 일비").
- **구어체 질문 12개**: 같은 내용을 직원이 실제로 물을 법한 말투로 바꿨습니다("하루 용돈 같은 거 얼마 나와요?"). 규정 용어와 겹치는 단어가 적습니다.

In [ ]:
import numpy as np
K_LIST = (1, 3, 5)
# 구어체 질문 12개: (질문, 정답 청크 id 집합)
casual = [("과장이 지방 출장 가면 하루 용돈 같은 거 얼마 나와요?", {"s2_t1_domestic"}),
          ("부장님 호텔비는 하룻밤 어디까지 회사가 내줘?", {"s2_t1_domestic"}),
          ("내 차 끌고 출장 가면 기름값은 어떻게 쳐줘?", {"s2_text"}),
          ("유럽 출장 가는 차장은 하루에 달러로 얼마 받아?", {"s3_t2_overseas"}),
          ("와이프가 아이를 낳으면 회사 며칠 쉴 수 있어?", {"s4_t4_family_leave"}),
          ("입사 5년 차면 쉬는 날이 1년에 며칠이야?", {"s4_t3_annual_leave"}),
          ("영어 학원비 회사에서 대 줘?", {"s5_t6_education"}),
          ("출장 다녀와서 영수증 처리는 언제까지 끝내야 돼?", {"s6_text"}),
          ("대학원 다니면 학비 지원받고 바로 퇴사해도 돼?", {"s5_t6_education", "s5_text"}),
          ("결혼하면 회사에서 축의금 주나요?", {"s5_t5_condolence"}),
          ("비행기 표랑 비자 비용도 회사 부담이야?", {"s3_text"}),
          ("이 규정은 몇 번이나 고쳐졌어?", {"s1_t0_history"})]
SETS = {"규정 질문 13": ([q["q"] for q in evalset], [q["gold"] for q in evalset]),
        "구어체 질문 12": ([c[0] for c in casual], [c[1] for c in casual])}

def gold_rank(order, gold):
    return min(order.index(g) + 1 for g in gold)

def scores(ranked, golds, ks=K_LIST):
    """ranked: 질문마다 청크 id를 점수 높은 순으로 정렬한 리스트 → hit@k, MRR"""
    r = [gold_rank(o, g) for o, g in zip(ranked, golds)]
    out = {f"hit@{k}": round(float(np.mean([x <= k for x in r])), 2) for k in ks}
    out["MRR"] = round(float(np.mean([1 / x for x in r])), 2)
    return out

def dense_rank(key, qs, **kw):
    D = embed_docs(key, chunk_texts, **kw)
    Q = embed_query(key, qs, **kw)
    S = models[key].similarity(Q, D).numpy()       # (질문 수 × 13) 코사인 유사도
    return [[chunk_ids[j] for j in np.argsort(-row)] for row in S]

ranked, rows = {}, []
for key in models:
    t0 = time.time()
    for set_name, (qs, golds) in SETS.items():
        ranked[(key, set_name)] = dense_rank(key, qs)
        rows.append({"model": key, "질문 세트": set_name, **scores(ranked[(key, set_name)], golds)})
    info[key]["임베딩 51개(초)"] = round(time.time() - t0, 2)    # 청크 13×2 + 질문 25
res = pd.DataFrame(rows).pivot(index="model", columns="질문 세트")
res = res.reorder_levels([1, 0], axis=1).sort_index(axis=1, level=0)
display(res)

# 질문별로 정답 청크가 몇 등에 나왔는지 (1 = 맨 위). 빨갈수록 놓친 것
qs, golds = SETS["구어체 질문 12"]
per_q = pd.DataFrame({key: [gold_rank(o, g) for o, g in zip(ranked[(key, "구어체 질문 12")], golds)] for key in models},
                     index=[q[:26] for q in qs])
display(per_q.style.background_gradient(cmap="Reds", vmin=1, vmax=5))

💡 **해석** — 강사 환경 측정값(CPU와 GPU의 임베딩 결과는 같습니다):

| 모델 | 규정 질문 hit@1 | hit@3 | MRR | 구어체 질문 hit@1 | hit@3 | MRR |
|---|---|---|---|---|---|---|
| e5-small | 0.92 | 1.00 | 0.96 | 0.67 | 0.92 | 0.81 |
| **KURE-v1** | **0.92** | 1.00 | **0.96** | **0.83** | **1.00** | **0.90** |
| EG2 | 0.85 | 1.00 | 0.92 | 0.67 | 1.00 | 0.83 |

- **규정 질문**에서는 세 모델 모두 hit@3이 1.00입니다. 청크가 13개뿐이고 질문이 규정 용어를 그대로 쓰기 때문입니다. 이렇게 쉬운 평가셋으로는 모델 차이가 안 보입니다.
- **구어체 질문**에서 차이가 납니다. KURE-v1이 hit@1 0.83으로 가장 좋고, e5와 EG2는 0.67입니다. e5는 한 문항을 3위 밖으로 놓쳤습니다(hit@3 0.92). KURE만 1위로 찾은 질문은 '부장님 호텔비'(숙박비 상한), '와이프가 아이를 낳으면'(배우자 출산), '이 규정은 몇 번이나 고쳐졌어'(개정 이력)입니다. 한국어 검색 데이터로 파인튜닝했다는 카드 설명과 맞아떨어지는 결과입니다.
- EG2는 한국어 성능이 카드에 명시되어 있지 않은데, 이 측정에서는 e5와 비슷하고 KURE보다 낮았습니다. EG2의 강점은 다음 노트북의 **이미지 검색**입니다.
- 질문 한 개가 hit@1을 0.08씩 움직이는 작은 평가셋이라는 점을 잊지 마세요.

## 4. 속도·크기 한눈에 🔵
검색 품질만큼 중요한 것이 **비용**입니다. 문서가 수만 개면 임베딩 시간이 그대로 곱해집니다.

In [ ]:
cost_tbl = pd.DataFrame(info).T[["파라미터(M)", "차원", "fp32 메모리(GB)", "로드(초)", "임베딩 51개(초)"]]
cost_tbl["벡터 1만 개 저장(MB)"] = (cost_tbl["차원"].astype(int) * 4 * 10_000 / 1024**2).round(1)
display(cost_tbl)

💡 **해석** — 강사 노트북(CPU)에서 텍스트 51개 임베딩은 e5 2초, EG2 20초, KURE 27초였습니다(다른 작업과 겹쳐 실행되면 더 걸립니다). T4 GPU에서는 보통 세 모델 모두 1~2초 안쪽입니다. CPU에서 e5-small은 KURE보다 10배 이상 빨랐습니다(코랩 T4에서는 0.27초 대 1.32초로 약 5배). 메모리는 5분의 1입니다.
차원이 크면 벡터DB 저장 공간과 검색 계산량이 늘어납니다. KURE(1024d)는 e5(384d)보다 저장 공간이 2.7배입니다. **"제일 좋은 모델"이 아니라 "품질 대비 비용이 맞는 모델"** 을 고르는 것이 실무입니다.

## 5. (🟠 심화) API 임베딩 한 번 써 보기
OpenAI `text-embedding-3-small`(1536차원, 1M 토큰당 $0.02)로 같은 평가를 해 봅니다. 바로 아래 키 입력 셀을 먼저 실행합니다(다른 노트북과 같은 셀: 입력 즉시 키를 확인하고, 틀리면 최대 3회 다시 입력). 코랩은 노트북마다 런타임이 따로라서 다른 노트북에서 입력한 키는 여기로 넘어오지 않습니다. 이 심화를 건너뛰려면 두 셀 모두 실행하지 않습니다.
청크 13개 + 질문 25개는 약 4,000토큰이므로 비용은 0.1원 수준입니다.
결과는 **3절과 같은 청크·질문·채점 함수**로 계산해 3절 표에 한 줄(`API-3small`)로 붙입니다. OpenAI 임베딩은 길이가 1로 나오므로 내적이 곧 코사인 유사도입니다.

In [ ]:
# ⚙️ OpenAI API 키 입력 — 수업에서 나눠 준 키를 붙여 넣으세요. (화면에 표시되지 않습니다)
# 규칙: 키를 코드에 직접 적지 말 것 · 출력하지 말 것 · 수업 후 키는 폐기됩니다.
# 입력한 키는 모델 조회 API(GET /v1/models/{모델} = client.models.retrieve)로 바로 확인합니다. 토큰을 쓰지 않아 비용이 없습니다.
import os, json, urllib.request, urllib.error
from getpass import getpass
MODEL_API = "gpt-6-luna"        # 수업용 소형 모델 ($0.10 / $0.50 per 1M tokens). 바뀌면 여기만 수정.

def check_key(key: str):
    """키 확인 → (상태, 안내). 키 문자열과 오류 본문은 출력하지 않는다(유형·상태 코드만)."""
    req = urllib.request.Request(f"https://api.openai.com/v1/models/{MODEL_API}",
                                 headers={"Authorization": f"Bearer {key}"})
    try:
        with urllib.request.urlopen(req, timeout=15):
            return "ok", "키 확인 완료"
    except urllib.error.HTTPError as e:
        try:
            err = json.loads(e.read().decode()).get("error", {})        # 판단에만 쓰고 출력하지 않는다
        except Exception:
            err = {}
        code, msg = err.get("code"), str(err.get("message", "")).lower()
        scope = "insufficient permissions" in msg or "missing scopes" in msg   # 권한 범위가 좁은 제한 키
        if e.code == 401 and not scope and code in ("invalid_api_key", None):
            return "bad", "키가 올바르지 않습니다(401). 빠진 글자·앞뒤 공백 없이 다시 붙여 넣으세요."
        if e.code in (401, 403):
            return "ok", f"키는 유효하지만 모델 조회 권한이 없습니다({e.code}). 실습은 진행하고, 호출이 막히면 강사에게 알려 주세요."
        if e.code == 404:
            return "ok", f"키는 유효하지만 모델 {MODEL_API}에 접근할 수 없습니다(404). 강사에게 알려 주세요."
        if e.code == 429:
            return "ok", "키는 유효합니다. 요청 한도에 걸렸습니다(429). 잠시 뒤 다시 실행하세요."
        return "err", f"확인 실패: HTTPError {e.code}. 잠시 뒤 이 셀을 다시 실행하세요."
    except Exception as e:
        return "err", f"확인 실패: {type(e).__name__}. 네트워크 연결을 확인하고 이 셀을 다시 실행하세요."

key = os.environ.get("OPENAI_API_KEY", "").strip()
for attempt in range(1, 4):
    if not key:
        key = getpass(f"OpenAI API 키 ({attempt}/3): ").strip()
    status, msg = check_key(key)
    print(msg)
    if status != "bad":
        break
    key = ""                                   # 틀린 키는 버리고 다시 입력받는다
if status == "bad":
    os.environ.pop("OPENAI_API_KEY", None)
    raise RuntimeError("키 확인 3회 실패: 강사에게 키를 다시 받은 뒤 이 셀을 다시 실행하세요.")
os.environ["OPENAI_API_KEY"] = key
print("모델:", MODEL_API)

In [ ]:
# 🟠 키가 없으면 건너뜁니다
try:
    from openai import OpenAI
    oa = OpenAI()
    r1 = oa.embeddings.create(model="text-embedding-3-small", input=chunk_texts)
    D = np.array([d.embedding for d in r1.data]); tok = r1.usage.total_tokens
    for set_name, (qs, golds) in SETS.items():
        r2 = oa.embeddings.create(model="text-embedding-3-small", input=qs); tok += r2.usage.total_tokens
        Q = np.array([d.embedding for d in r2.data])
        ranked[("API-3small", set_name)] = [[chunk_ids[j] for j in np.argsort(-row)] for row in Q @ D.T]
    print(f"차원 {D.shape[1]} · 토큰 {tok} · 비용 ${tok * 0.02 / 1e6:.6f}")

    # 3절 표에 API 한 줄을 붙여 나란히 비교
    rows = [{"model": key, "질문 세트": set_name, **scores(ranked[(key, set_name)], golds)}
            for key in [*models, "API-3small"] for set_name, (qs, golds) in SETS.items()]
    res_api = pd.DataFrame(rows).pivot(index="model", columns="질문 세트")
    display(res_api.reorder_levels([1, 0], axis=1).sort_index(axis=1, level=0))

    # 구어체 질문별 정답 순위 — 3절 표에 API 열 추가 (1 = 맨 위, 빨갈수록 놓친 것)
    qs, golds = SETS["구어체 질문 12"]
    per_q["API-3small"] = [gold_rank(o, g) for o, g in zip(ranked[("API-3small", "구어체 질문 12")], golds)]
    display(per_q.style.background_gradient(cmap="Reds", vmin=1, vmax=5))
except Exception as e:
    print("API 임베딩 건너뜀(키·네트워크 확인):", type(e).__name__, getattr(e, "status_code", ""))

💡 **해석** — 코랩 T4 실측입니다. 로컬 세 모델의 값은 3절 표와 같습니다.

| 모델 | 규정 질문 hit@1 | hit@3 | MRR | 구어체 질문 hit@1 | hit@3 | MRR |
|---|---|---|---|---|---|---|
| e5-small (로컬, 118M) | 0.92 | 1.00 | 0.96 | 0.67 | 0.92 | 0.81 |
| **KURE-v1** (로컬, 568M) | **0.92** | 1.00 | **0.96** | **0.83** | **1.00** | **0.90** |
| EG2 (로컬, 271M) | 0.85 | 1.00 | 0.92 | 0.67 | 1.00 | 0.83 |
| API-3small (OpenAI) | 0.85 | 1.00 | 0.91 | **0.50** | 0.83 | **0.64** |

- **규정 질문**: API도 hit@3이 1.00입니다. 3절에서 본 것처럼, 규정 용어를 그대로 쓰는 질문은 어느 모델이든 찾습니다. hit@1 0.85는 EG2와 같고 e5·KURE보다 한 문항 낮습니다.
- **구어체 질문**: API가 넷 중 가장 낮습니다. hit@1 0.50(12문항 중 6개)으로 KURE-v1(10개)보다 4문항 적고, 가장 작은 e5-small(8개)보다도 낮습니다. 정답이 3위 밖으로 밀린 질문도 2개입니다('부장님 호텔비' 5위, '와이프가 아이를 낳으면' 6위).
- 질문별 순위표에서 API가 1위를 놓친 6문항은 '하루 용돈'(3위), '부장님 호텔비'(5위), '기름값'(3위), '와이프가 아이를 낳으면'(6위), '축의금'(3위), '몇 번이나 고쳐졌어'(3위)입니다. 이 중 '하루 용돈'과 '기름값'은 로컬 세 모델이 모두 1위로 찾은 질문입니다. 반대로 '영어 학원비'는 API가 1위, KURE가 3위였습니다. 같은 구어체라도 모델마다 강한 질문이 다릅니다.
- 참고로 KURE-v1 카드(저자 보고, 이론2 p68)에서도 OpenAI의 더 큰 모델 `text-embedding-3-large`가 Recall@10 0.729로 KURE-v1(0.797)보다 낮게 보고되었습니다. 데이터셋과 지표가 달라 우리 값과 직접 비교하지는 않습니다. **유료 API라고 한국어 검색을 항상 더 잘하는 것은 아닙니다.**
- **비용과 운영**: 이번 호출은 4,174토큰, $0.000083(약 0.1원)이었습니다. GPU도 모델 다운로드도 필요 없습니다. 대신 세 가지를 따져야 합니다.
  - 청크 본문이 OpenAI 서버로 나갑니다. 사내 데이터 정책을 먼저 확인합니다(이론2 p65).
  - 1536차원이라 벡터 1만 개 저장에 58.6MB가 듭니다(4절과 같은 계산: 차원 × 4바이트 × 1만 개). 4절 표의 KURE(39.1MB)의 1.5배, e5(14.6MB)의 4배입니다.
  - 검색할 때마다 질문을 임베딩하는 API 호출이 한 번씩 네트워크를 탑니다.
- 질문 12개짜리 평가라 한 문항이 hit@1을 0.08씩 움직입니다. 그래도 "유료 API로 바꾸면 검색이 좋아진다"고 가정하면 안 된다는 점은 분명히 보여 줍니다. 모델을 바꿀 때는 이 셀처럼 **같은 평가셋·같은 채점으로** 재고 정합니다.

## 6. 바꿔서 실행 🟡
1. **접두어를 빼면?** 아래 셀은 e5의 `"query: "`/`"passage: "`, EG2의 `prompt_name`을 빼고 다시 평가합니다. 모델 카드는 둘 다 "붙여야 한다"고 합니다. 우리 데이터에서도 그런지 확인하세요.
2. **내 질문 추가** — `my_q`에 규정집에 대한 질문을, `my_gold`에 정답 청크 id(맨 위 표의 `id`)를 넣고 세 모델이 몇 등에 찾는지 보세요.

In [ ]:
# 1) 접두어·프롬프트 없이 평가
def rank_plain(key, qs):
    m = models[key]
    D = m.encode(chunk_texts, normalize_embeddings=True)      # "passage: " / prompt_name 없음
    Q = m.encode(qs, normalize_embeddings=True)               # "query: " / prompt_name 없음
    return [[chunk_ids[j] for j in np.argsort(-row)] for row in m.similarity(Q, D).numpy()]

rows = []
for key in ["e5-small", "EG2"]:
    for set_name, (qs, golds) in SETS.items():
        rows.append({"model": key, "질문 세트": set_name, "접두어": "있음", **scores(ranked[(key, set_name)], golds)})
        rows.append({"model": key, "질문 세트": set_name, "접두어": "없음", **scores(rank_plain(key, qs), golds)})
display(pd.DataFrame(rows).set_index(["model", "질문 세트", "접두어"]))

💡 **해석** — 강사 환경에서는 **접두어를 빼도 규정 질문의 hit@k는 같았고(EG2 MRR만 0.92→0.91), 구어체 질문은 오히려 올랐습니다**(e5 hit@1 0.67→0.83, EG2 0.67→0.83). 카드의 권장과 다른 결과입니다.
- 이유 후보: 청크 13개·질문 25개는 너무 작아 몇 문항의 순위 변화가 그대로 점수가 됩니다. 접두어 효과는 수만 개 문서에서 비슷한 후보가 많을 때 커집니다.
- 그래도 실무 기본값은 **카드대로 붙이기**입니다. 빼기로 했다면 이 셀처럼 **우리 평가셋으로 재서 근거를 남기세요**. 그리고 문서와 질문은 반드시 **같은 방식**으로 넣어야 합니다. 문서는 접두어 없이 저장하고 질문만 붙이면 점수가 틀어집니다.

In [ ]:
# 2) 내 질문 추가 — 바꿔 보세요
my_q = "해외 출장 갈 때 여행자보험료도 회사가 내 주나요?"
my_gold = "s3_text"
for key in models:
    order = dense_rank(key, [my_q])[0]
    print(f"{key:<9} 정답 {my_gold} → {order.index(my_gold)+1}등 · 1~3등: {order[:3]}")

### 🤔 생각해보기

**벤치마크 1등이 우리 문서에서도 1등인가?** 이 노트북의 결과로 답해 보세요. 그리고 실제 회사라면 모델을 고르기 전에 무엇을 준비해야 할까요?

<details><summary><b>🤔 생각해보기 — 해설 보기(클릭)</b></summary>

- 아닐 수 있습니다. 공개 벤치(MTEB-ko, KURE 보고표)는 위키·뉴스 같은 일반 문서로 잽니다. 우리 규정집은 **표·숫자·사내 용어**가 많아 결과 순위가 바뀔 수 있습니다. 이 노트북에서도 모델 간 차이가 공개 벤치의 차이와 같은 방향인지 위 표로 확인해 보세요.
- 질문이 13개뿐이라 한 문항이 hit@k를 0.08씩 움직입니다. 작은 평가셋의 차이는 **잡음일 수 있습니다**.
- 그래서 실무에서는 먼저 **우리 질문 50~200개 + 정답 근거 id**로 된 평가셋을 만들고, 후보 모델 2~3개를 같은 코드로 잽니다. 오늘 쓴 `hanbit_qa.json`이 그 축소판입니다.
- 유료 API도 예외가 아닙니다. 5절에서 `text-embedding-3-small`은 구어체 질문에서 넷 중 가장 낮았습니다.
- 품질이 비슷하면 **속도·메모리·차원(저장 비용)·라이선스·데이터 정책**으로 고릅니다.

</details>

---
### ✅ 이 노트북에서 가져갈 것
- 임베딩 모델은 사용법(접두어·`prompt_name`)을 지켜야 제 성능이 나온다. 빠뜨리면 조용히 나빠진다.
- 모델 선택은 **우리 질문 + 정답 근거**로 hit@k를 재서 한다. 벤치 순위는 참고만.
- API 임베딩(`text-embedding-3-small`)은 서버 없이 거의 공짜로 쓰지만, 이 평가셋의 구어체 질문에서는 로컬 모델 3개보다 낮았다. 문서가 외부로 나간다는 점도 따진다.

> 다음 노트북: **10_multimodal_embeddings — OCR 없이 스캔 페이지를 한국어로 찾기**